# Paper-Ready Metric Plots

PSNR / SSIM vs transmission budget. Error bars = 95% CI of the mean: `mean ± 1.96 × (σ/√n)`.

**Run from `dlapisgs-utility/`:**
```bash
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=120 \
    plotting/paper_plot_metrics.ipynb
```

In [ ]:
import sys
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")

In [ ]:
# ------------------------------- setting start ------------------------------ #
color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
errorbar_color = "#3A3A3A"

# font
csfont = {'family': 'serif', 'serif': ['Times New Roman', 'Times'], 'size': 23}

# errorbar plot size
err_lw       = 1.5
err_capsize  = 4
err_capthick = 1.5

# figure size
figsize = (6.4, 4.8)

# set theme first, then rc — so seaborn doesn't clobber the font size
sns.set_theme(style="ticks", font="Times New Roman")
plt.rc('text', usetex=True)
plt.rc('font', **csfont)
plt.rcParams['text.latex.preamble'] = r'\usepackage{mathptmx}'
# -------------------------------- setting end ------------------------------- #

In [ ]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# Paths relative to dlapisgs-utility/ (set by the working-directory cell below)
SUMMARY_CSV = "output/0603/exp1_1600_vol_modes/summary_all.csv"
OUT_DIR     = "plotting/paper"
GROUP_BY    = "weight_mode"
EXCLUDE_KEYS = ["det_gamma_over_d2"]

# budget percentages used in the sweep (must match distinct budgets per scene in order)
BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

# paper-specific label/marker/color overrides (merged on top of plot_metrics defaults)
color_palette = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf"]

KEY_CONFIG = {
    # weight modes
    "screen_area":      {"label": "Screen Area",                   "marker": "o", "color": color_palette[4]},
    "volume_over_d2":   {"label": r"Vol/$d^2$ (view-dep.)",        "marker": "D", "color": color_palette[3]},
    "volume":           {"label": "Volume (view-indep.)",          "marker": "^", "color": color_palette[2]},
    "random":           {"label": "Random (control)",              "marker": "x", "color": color_palette[7]},
    # schemes
    "vd_lod":           {"label": "VD+LOD (baseline)",             "marker": "s", "color": color_palette[0]},
    "vd_lod_w":         {"label": "VD+LOD+W",                      "marker": "^", "color": color_palette[1]},
    "ml":               {"label": "ML (ours)",                     "marker": "P", "color": color_palette[5]},
    "oracle_loo":       {"label": "Oracle LOO (upper bound)",       "marker": "*", "color": color_palette[9]},
    # packing conditions (exp3)
    "prog_cull":        {"label": "Progressive (frustum-cull)",    "marker": "o", "color": color_palette[0]},
    "prog_no_cull":     {"label": "Progressive (no culling)",      "marker": "^", "color": color_palette[2]},
    "tile_strict":      {"label": "Tile-strict",                   "marker": "s", "color": color_palette[1]},
}

KEY_ORDER = {
    "weight_mode": ["screen_area", "volume_over_d2", "volume", "random"],
    "scheme":      ["vd_lod", "vd_lod_w", "ml", "oracle_loo"],
    "condition":   ["prog_cull", "prog_no_cull", "tile_strict"],
}

DPI = 300

In [ ]:
import os, sys

# cd to dlapisgs-utility/ regardless of where nbconvert was invoked
_cwd = Path(os.getcwd())
_root = None
_candidates = [_cwd] + list(_cwd.parents) + [Path(p) for p in sys.path]
for _candidate in _candidates:
    if (_candidate / "utility_calculation.py").exists():
        _root = _candidate
        break
if _root is None:
    try:
        _root = Path(__file__).resolve().parent.parent
    except NameError:
        _root = _cwd
os.chdir(_root)
print(f"cwd: {Path.cwd()}")

In [ ]:
# shared data-pipeline: single source of truth with experiments/plot_metrics.py
sys.path.insert(0, str(Path.cwd()))
from experiments.plot_metrics import (
    _apply_budget_labels, _aggregate, _resolve_order_and_labels,
    _data_ylim, _bk_sort, PSNR_SATURATION_DB,
)

In [ ]:
summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()

# convert to list-of-dicts (plot_metrics format) and attach per-scene budget labels
rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")

In [ ]:
agg    = _aggregate(rows, GROUP_BY)
pm_order, _ = _resolve_order_and_labels(agg, GROUP_BY)

# paper ordering: prefer KEY_ORDER override, fall back to plot_metrics order
preferred = KEY_ORDER.get(GROUP_BY, pm_order)
order = [k for k in preferred if k in agg] + [k for k in pm_order if k not in preferred and k in agg]

sample_key = order[0]
sample_bk  = sorted(agg[sample_key], key=_bk_sort)[0]
print(f"agg sample  {GROUP_BY}={sample_key!r}  budget={sample_bk}:")
print(agg[sample_key][sample_bk])

In [ ]:
def plot_metric(agg, order, metric, ylabel, out_stem, out_dir):
    fallback_markers = ["s", "^", "D", "o", "v", "P", "X"]

    fig, ax = plt.subplots(figsize=figsize)
    all_means = []

    for i, key in enumerate(order):
        if key not in agg:
            continue
        cfg    = KEY_CONFIG.get(key, {})
        label  = cfg.get("label",  key)
        marker = cfg.get("marker", fallback_markers[i % len(fallback_markers)])
        color  = cfg.get("color",  color_palette[i % len(color_palette)])

        bks = sorted(agg[key], key=_bk_sort)
        xs  = [_bk_sort(bk) for bk in bks]
        ys  = [agg[key][bk][f"{metric}_mean"] for bk in bks]
        es  = [agg[key][bk][f"{metric}_ci95"]  for bk in bks]
        all_means.extend(ys)

        ax.errorbar(xs, ys, yerr=es,
                    marker=marker, color=color, linewidth=2, markersize=8,
                    capsize=err_capsize, elinewidth=err_lw, capthick=err_capthick,
                    label=label)

    ax.set_xlabel(r"Budget (\% of scene)")
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\%"))
    ax.set_ylim(*_data_ylim(all_means, metric))
    ax.legend(loc="upper left", framealpha=0.9)

    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)

    fig.tight_layout()
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{out_stem}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(out_dir / f"{out_stem}.eps", format="eps", bbox_inches="tight")
    print(f"Wrote {out_dir}/{out_stem}.{{png,eps}}")
    plt.close(fig)

In [ ]:
out_dir = Path(OUT_DIR)
plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
print("Done.")